In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Lab 2: Simulating Decentralized Agreement and Control\n",
    "**Course:** Blockchain and its Application  \n",
    "**Instructor:** Vishnu Suresh"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 1: Trust model and quorum approval"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Step 1: Define the participants and proposal\n",
    "validators = [\n",
    "    \"University A\",\n",
    "    \"University B\",\n",
    "    \"University C\",\n",
    "    \"University D\",\n",
    "    \"University E\",\n",
    "]\n",
    "\n",
    "proposal = {\n",
    "    \"student\": \"Marta Kowalska\",\n",
    "    \"credential\": \"Blockchain Fundamentals\",\n",
    "    \"issuer\": \"University A\",\n",
    "}\n",
    "\n",
    "# Step 2: Define the local validation rule\n",
    "def validate_proposal(proposal):\n",
    "    required = {\"student\", \"credential\", \"issuer\"}\n",
    "    missing = required - set(proposal)\n",
    "    if missing:\n",
    "        return False, f\"Missing fields: {sorted(missing)}\"\n",
    "    if proposal[\"issuer\"] not in validators:\n",
    "        return False, \"Issuer is not a consortium member\"\n",
    "    return True, \"Proposal follows the rules\"\n",
    "\n",
    "# Step 3: Collect votes and apply the quorum\n",
    "def quorum_result(votes, threshold=4):\n",
    "    approvals = sum(vote is True for vote in votes.values())\n",
    "    responding = sum(vote is not None for vote in votes.values())\n",
    "    return approvals >= threshold, approvals, responding\n",
    "\n",
    "votes = {\n",
    "    validator: validate_proposal(proposal)[0]\n",
    "    for validator in validators\n",
    "}\n",
    "\n",
    "print(\"Votes:\", votes)\n",
    "print(\"Quorum result (Accepted, Approvals, Responding):\", quorum_result(votes))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 2: Crash and Byzantine faults"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Step 1: Simulate one crashed validator (stops responding)\n",
    "crash_votes = votes.copy()\n",
    "crash_votes[\"University E\"] = None\n",
    "print(\"Crash votes:\", crash_votes)\n",
    "print(\"Crash quorum:\", quorum_result(crash_votes))\n",
    "\n",
    "# Step 2: Simulate one dishonest validator (rejects a valid proposal)\n",
    "dishonest_votes = votes.copy()\n",
    "dishonest_votes[\"University E\"] = False\n",
    "print(\"\\nDishonest votes:\", dishonest_votes)\n",
    "print(\"Dishonest quorum:\", quorum_result(dishonest_votes))\n",
    "\n",
    "# Step 3: Submit an invalid proposal with one malicious approval\n",
    "invalid_proposal = {\n",
    "    \"student\": \"Marta Kowalska\",\n",
    "    \"credential\": \"Blockchain Fundamentals\",\n",
    "    \"issuer\": \"Unknown University\",\n",
    "}\n",
    "\n",
    "invalid_votes = {\n",
    "    validator: validate_proposal(invalid_proposal)[0]\n",
    "    for validator in validators\n",
    "}\n",
    "# Malicious validator approves the invalid proposal\n",
    "invalid_votes[\"University E\"] = True\n",
    "\n",
    "print(\"\\nInvalid proposal votes:\", invalid_votes)\n",
    "print(\"Invalid proposal quorum:\", quorum_result(invalid_votes))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 3: Network partitions, safety, and liveness"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Step 1: Test a three-two partition with threshold 4\n",
    "def has_quorum(voter_set, threshold):\n",
    "    return len(set(voter_set)) >= threshold\n",
    "\n",
    "left_side = {\"University A\", \"University B\", \"University C\"}\n",
    "right_side = {\"University D\", \"University E\"}\n",
    "\n",
    "print(\"Left at threshold 4:\", has_quorum(left_side, 4))\n",
    "print(\"Right at threshold 4:\", has_quorum(right_side, 4))\n",
    "\n",
    "# Step 2: Lower the threshold to 3 and add equivocation (contradictory votes)\n",
    "proposal_x_voters = {\"University A\", \"University B\", \"University E\"}\n",
    "proposal_y_voters = {\"University C\", \"University D\", \"University E\"}\n",
    "\n",
    "print(\"\\nX at threshold 3:\", has_quorum(proposal_x_voters, 3))\n",
    "print(\"Y at threshold 3:\", has_quorum(proposal_y_voters, 3))\n",
    "print(\"Voted for both:\", proposal_x_voters & proposal_y_voters)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Part 3 Explanation: Safety vs. Liveness\n",
    "* **Threshold 4 of 5:** During a 3–2 network partition, neither side reaches the required 4 votes. The ledger pauses without finalizing any proposal. This protects **safety** (preventing conflicting decisions) at the cost of **liveness** (the system cannot make progress until communication is restored).\n",
    "* **Threshold 3 of 5:** The 3-node side can make progress (higher **liveness**), but a single equivocating validator (University E) can vote for two conflicting proposals and allow both groups to reach quorum simultaneously, violating **safety**."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 4: Sybil resistance"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Step 1: Count identities without membership control\n",
    "def simple_majority(votes):\n",
    "    approvals = sum(votes.values())\n",
    "    return approvals > len(votes) / 2\n",
    "\n",
    "honest_votes = {validator: False for validator in validators}\n",
    "fake_votes = {f\"Fake identity {number}\": True for number in range(1, 21)}\n",
    "\n",
    "open_votes = {**honest_votes, **fake_votes}\n",
    "print(\"Identity count:\", len(open_votes))\n",
    "print(\"Attacker wins (open):\", simple_majority(open_votes))\n",
    "\n",
    "# Step 2: Apply permissioned membership filtering\n",
    "authorized = set(validators)\n",
    "authorized_votes = {\n",
    "    identity: vote\n",
    "    for identity, vote in open_votes.items()\n",
    "    if identity in authorized\n",
    "}\n",
    "\n",
    "print(\"\\nAuthorized identities:\", len(authorized_votes))\n",
    "print(\"Attacker wins (filtered):\", simple_majority(authorized_votes))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 5: Effective control and unequal voting power"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Step 1: Add the coalition function to find minimum coalition reaching threshold\n",
    "def minimum_coalition(weights, threshold):\n",
    "    ordered = sorted(weights.items(), key=lambda item: item[1], reverse=True)\n",
    "    coalition = []\n",
    "    total = 0\n",
    "    for validator, weight in ordered:\n",
    "        coalition.append(validator)\n",
    "        total += weight\n",
    "        if total >= threshold:\n",
    "            return coalition, total\n",
    "    return coalition, total\n",
    "\n",
    "# Step 2: Compare equal and concentrated power (threshold = 67)\n",
    "equal_weights = {\n",
    "    \"A\": 20, \"B\": 20, \"C\": 20,\n",
    "    \"D\": 20, \"E\": 20\n",
    "}\n",
    "\n",
    "concentrated_weights = {\n",
    "    \"A\": 40, \"B\": 25, \"C\": 15,\n",
    "    \"D\": 10, \"E\": 10\n",
    "}\n",
    "\n",
    "print(\"Equal:\", minimum_coalition(equal_weights, 67))\n",
    "print(\"Concentrated:\", minimum_coalition(concentrated_weights, 67))\n",
    "\n",
    "# Step 3: Create an extreme case (Validator A = 70, remaining 30 divided among others)\n",
    "extreme_weights = {\n",
    "    \"A\": 70, \"B\": 10, \"C\": 10,\n",
    "    \"D\": 5, \"E\": 5\n",
    "}\n",
    "\n",
    "print(\"Extreme:\", minimum_coalition(extreme_weights, 67))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Part 5 Explanation: Extreme Voting Power\n",
    "When Validator A's weight is increased to 70, A alone reaches the 67 threshold (`(['A'], 70)`). Even though the network still displays five validators, effective control is completely centralized in a single actor. Therefore, validator count alone does not accurately describe decentralization; the distribution of voting weights must be measured."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 6: Final Design Proposal (Scenario 1: Five Universities Sharing Micro-Credentials)\n",
    "\n",
    "* **Who may submit records and who validates them:** Only authenticated consortium universities (the 5 member institutions) may submit credential proposals, and all 5 designated university nodes act as validators.\n",
    "* **Who may read the ledger and who governs membership or upgrades:** Employers, students, and the public may read and verify credentials (public read access), while consortium governance (adding/removing universities or upgrading protocol rules) is managed jointly by the founding universities via off-chain agreements and on-chain supermajority votes.\n",
    "* **Which failures or collusion the design must tolerate:** With $N=5$ validators and a threshold of $4$, the system tolerates $1$ crash fault (offline node) or $1$ Byzantine/malicious validator ($f=1$) without compromising ledger integrity.\n",
    "* **How the design resists fake identities:** Through permissioned membership and cryptographic identity verification (digital signatures/PKI). Unknown keys are automatically rejected by the validation rules, preventing Sybil attacks.\n",
    "* **What quorum, fork-choice, or finality rule determines acceptance:** A strict 4-of-5 (80%) quorum rule determines acceptance. Because $4 + 4 > 5$, two conflicting proposals cannot both reach quorum simultaneously (assuming at most 1 equivocator), providing deterministic instant finality without forks.\n",
    "* **Where effective control could become concentrated:** Control could concentrate if universities are assigned unequal voting weights (e.g., based on student population or funding) or if a single third-party cloud provider hosts the validator infrastructure for multiple universities.\n",
    "* **Whether a simpler signed database would meet the same need:** A conventional centralized signed database could store credentials, but it would introduce a single point of failure and unilateral administrative control. A shared consortium ledger ensures that no single university can retroactively alter, censor, or forge records without the consensus of at least three other institutions."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.13.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}